# 01 · Data understanding

**Goal:** know exactly what the data is before modelling it.

| | |
|---|---|
| **What** | Download M5 (Walmart unit sales, 2011-01-29 to 2016-05-22), validate it, and build 211 weekly series. |
| **Why** | Every later number depends on the data being correct and correctly aggregated. A silent error here (for example treating "not yet launched" as "zero demand") would bias every model. |
| **Alternatives** | UCI Online Retail II (one UK web shop, about 2 years, needs heavy transaction cleaning), Rossmann/Favorita (need a Kaggle login, which was not available). See `DECISIONS.md` D-03. |
| **Result** | 30,490 item x store series, 1,941 days, 0 missing cells, 0 duplicates, 0 negatives. |
| **Meaning** | The data is clean at the cell level; the real modelling challenges are *zeros*, *launch dates* and *seasonality*, not dirty records. |

In [1]:
import os, sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
def show(fig): display(Image(filename=f"reports/figures/{fig}"))
def J(path): return json.load(open(path))

## 1. Raw files

`make data` runs `src/data/download.py` (HTTP, with a sparse git clone as fallback, SHA-256 verified) and then `src/data/prepare.py`.
Raw CSVs live in `data/raw/m5/` and are **git-ignored** (about 490 MB).

In [2]:
raw = ROOT / "data/raw/m5"
cal = pd.read_csv(raw / "calendar.csv")
prices = pd.read_csv(raw / "sell_prices.csv", nrows=5)
sales_head = pd.read_csv(raw / "sales_train_evaluation.csv", nrows=3, usecols=range(10))
display(cal.head(3)); display(prices); display(sales_head)

,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,NaN,NaN,NaN,NaN,0,0,0


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,d_5
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0


**Column meanings**
- `sales_train_evaluation.csv`: one row per item x store; `d_1 … d_1941` = units sold per day; ids give `item_id`, `dept_id` (7), `cat_id` (FOODS / HOBBIES / HOUSEHOLD), `store_id` (10), `state_id` (CA / TX / WI).
- `calendar.csv`: maps `d_*` to dates; `wm_yr_wk` = Walmart fiscal week (Saturday to Friday); `event_name/type` (sporting, cultural, national, religious); `snap_CA/TX/WI` = days on which SNAP food-stamp benefits are paid.
- `sell_prices.csv`: weekly shelf price per store x item x week. A row exists **only when the item is on sale in that store**.

## 2. Validation profile (`reports/data_profile.json`)

In [3]:
prof = J("reports/data_profile.json")
for k in ["files", "date_range_daily", "missing_values", "duplicates", "negative_values", "zero_inflation", "calendar_gaps_days"]:
    print(f"{k:22s}", prof[k])
print("daily total minimum:", prof["daily_total_units"]["min"], "on", prof["daily_total_units"]["min_date"])
print("days < 10% of median:", prof["daily_total_units"]["days_below_10pct_of_median"])

files                  {'sales_train_evaluation.csv': {'rows': 30490, 'cols': 1947}, 'calendar.csv': {'rows': 1969, 'cols': 13}, 'sell_prices.csv': {'rows': 6841121, 'cols': 4}}
date_range_daily       ['2011-01-29', '2016-05-22']
missing_values         {'sales_cells': 0, 'sales_id_cells': 0, 'sell_prices': {'store_id': 0, 'item_id': 0, 'wm_yr_wk': 0, 'sell_price': 0}, 'calendar_event_name_1_non_null_days': 158}
duplicates             {'sales_duplicate_ids': 0, 'sales_duplicate_rows_on_id_cols': 0, 'prices_duplicate_keys': 0, 'calendar_duplicate_dates': 0}
negative_values        {'sales_cells': 0, 'prices': 0}
zero_inflation         {'share_zero_cells_daily_item_store': 0.6799776584040612, 'share_zero_weeks_item_store_while_listed': 0.23904225518043867, 'share_zero_weeks_item_store_all': 0.3973527451149871}
calendar_gaps_days     0
daily total minimum: 11 on 2012-12-25
days < 10% of median: ['2011-12-25', '2012-12-25', '2013-12-25', '2014-12-25', '2015-12-25']


**Reading it**
- No missing values, duplicates, negative sales or calendar gaps, so no imputation is needed.
- 68% of *daily* item x store cells are zero. At this grain most demand is intermittent, which is one reason to aggregate to weeks.
- The five near-zero days are all **25 December**: stores are closed. These are *real* calendar effects, not data errors, so we keep them (the Christmas week is about 13% below its neighbours, see notebook 02).

## 3. "No sales" vs "no data": the listing mask

**What:** an item x store is considered *listed* in a week only if `sell_prices.csv` has a price for it.
**Why:** before launch, a zero means "the product did not exist", not "nobody wanted it". Feeding launch zeros to a model would teach it that demand was zero.
**Check:** if the mask is right, *no units* should ever be sold in an unlisted week.

In [4]:
print(json.dumps(prof["listing"], indent=1))

{
 "sales_in_unlisted_weeks_units": 0,
 "item_store_listed_from_week0": 10932,
 "item_store_with_internal_unlisted_weeks": 0,
 "median_first_listed_week": 22.0,
 "share_item_store_weeks_unlisted_after_launch": 0.0
}


**Result:** 0 units were sold in unlisted weeks, so the mask is exact. 10,932 item x stores are listed from week 0; the median launch is week 22; there are no relistings after a delisting, so SKU series simply start at their first listed week.
Zeros *while listed* (23.9% of listed item-store weeks) are either genuine zero demand **or unobserved stock-outs**. The data cannot tell these apart (censoring, see notebook 02).

## 4. Weekly series used for modelling

**Granularity = weekly** (Walmart weeks, Saturday to Friday): 277 complete weeks; the last 2 days are dropped so the final week is not a fake drop.
**Series set** (DECISIONS D-10): all 91 aggregates (total, 3 categories, 7 departments, 10 stores, 70 store x department) plus 120 SKU x store series stratified 30 per Syntetos-Boylan class, split by volume tercile.

In [5]:
meta = pd.read_csv("data/processed/series_meta.csv")
display(meta.level.value_counts().rename("n_series").to_frame())
print(json.dumps(prof["sku_sampling"], indent=1))
display(meta[meta.level == "sku_store"].sample_stratum.value_counts().sort_index().to_frame("n"))

,n_series
level,
sku_store,120
store_dept,70
store,10
department,7
category,3
total,1


{
 "eligible_item_store_series": 20366,
 "eligibility_rule": "listed >= 200 of 277 weeks and no unlisted gap after launch",
 "sbc_counts_in_eligible": {
  "smooth": 10360,
  "intermittent": 6951,
  "lumpy": 1781,
  "erratic": 1274
 }
}


,n
sample_stratum,
erratic/high,10
erratic/low,10
erratic/mid,10
intermittent/high,10
intermittent/low,10
intermittent/mid,10
lumpy/high,10
lumpy/low,10
lumpy/mid,10


**Why stratify the SKU sample?** A random draw from the 20,366 eligible SKUs would be about 51% smooth and contain few erratic items. Equal class sizes let us compare models *within* each demand type, which is the core of notebook 05.
**Eligibility (listed at least 200 of 277 weeks):** guarantees at least 2 seasonal cycles before the first forecast origin. The cost is that new-product cold starts are excluded (a stated limitation).

## 5. Limitations of this dataset for real retail demand
1. **No apparel.** Walmart FOODS / HOUSEHOLD / HOBBIES; fashion has shorter life-cycles, size/colour variants and markdown seasons.
2. **Sales ≠ demand.** Stock-outs are not recorded, so observed sales understate demand (censoring).
3. **No promotion flags.** Prices change rarely (1.4% of SKU-weeks) and there is no promo calendar.
4. **No lead times, costs or inventory positions.** The inventory part is a simulation under stated assumptions.
5. **US-specific calendar** (SNAP, Super Bowl, Thanksgiving). An Indian retailer would need Diwali, wedding season, EOSS sales, etc.